# Data cleaning
We want to process the data to store it into the Silver Layer. We already know what we want for the gold table: a Stock Summary. Because we already have the daily summary, we will do a monthly stock summary.

First we want to retrieve the bronze tables.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, DecimalType, IntegerType, DoubleType
from pyspark.sql.window import Window

stock_table = spark.read.table('jrvs_dlt.01_bronze.stock_data')
meta_table = spark.read.table('jrvs_dlt.01_bronze.stock_meta_data')
company_table = spark.read.table('jrvs_dlt.01_bronze.company_data')
display(stock_table)
display(meta_table)
display(company_table)

date,open,high,low,close,volume
2026-02-10,191.38,192.48,188.12,188.54,136764825
2026-02-09,184.26,193.66,183.95,190.04,196387351
2026-02-06,176.69,187.0,174.6,185.41,231346241
2026-02-05,174.925,176.815,171.03,171.88,206312890
2026-02-04,179.46,179.58,171.91,174.19,207014116
2026-02-03,186.24,186.27,176.23,180.34,202006430
2026-02-02,187.2,190.3,184.88,185.61,165794054
2026-01-30,191.21,194.49,189.47,191.13,179489463
2026-01-29,191.34,193.48,186.06,192.51,171764375
2026-01-28,191.27,192.35,189.84,191.52,148552677


information,symbol,last_refreshed,output_size,time_zone
"Daily Prices (open, high, low, close) and Volumes",NVDA,2026-02-10,Compact,US/Eastern


200DayMovingAverage,50DayMovingAverage,52WeekHigh,52WeekLow,Address,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongBuy,AnalystRatingStrongSell,AnalystTargetPrice,AssetType,Beta,BookValue,CIK,Country,Currency,Description,DilutedEPSTTM,DividendDate,DividendPerShare,DividendYield,EBITDA,EPS,EVToEBITDA,EVToRevenue,ExDividendDate,Exchange,FiscalYearEnd,ForwardPE,GrossProfitTTM,Industry,LatestQuarter,MarketCapitalization,Name,OfficialSite,OperatingMarginTTM,PEGRatio,PERatio,PercentInsiders,PercentInstitutions,PriceToBookRatio,PriceToSalesRatioTTM,ProfitMargin,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenuePerShareTTM,RevenueTTM,Sector,SharesFloat,SharesOutstanding,Symbol,TrailingPE
170.11,183.81,212.18,86.6,"2788 SAN TOMAS EXPRESSWAY, SANTA CLARA, CA, UNITED STATES, 95051",48,3,1,12,0,253.62,Common Stock,2.314,4.892,1045810,USA,USD,"NVIDIA Corporation is a premier American multinational technology company based in Santa Clara, California, celebrated for its groundbreaking advancements in graphics processing units (GPUs) aimed at gaming and professional markets. As a frontrunner in artificial intelligence and visual computing, NVIDIA plays a vital role in developing transformative technologies, including System on a Chip (SoC) products that enhance mobile computing and revolutionize the automotive sector, particularly in autonomous driving. With a robust and diversified portfolio that encompasses gaming, data centers, and AI infrastructure, NVIDIA stands at the forefront of the evolving tech landscape, consistently driving innovation and performance to meet the growing demands of its global clientele.",4.02,2025-12-26,0.04,0.0002,112696001000,4.02,38.06,24.22,2025-12-04,NASDAQ,January,24.51,131092996000,SEMICONDUCTORS,2025-10-31,4590383137000,NVIDIA Corporation,https://www.nvidia.com,0.632,0.709,46.9,4.330,69.586,38.54,24.53,0.53,0.667,0.625,0.535,1.074,7.67,187141997000,TECHNOLOGY,23330916000,24305000000,NVDA,46.9


### Stock Table Transformations
- Core metrics: Open, High, Low, Close, Volume.
- Calculated fields such as daily returns, price change %.

In [0]:
stock_table = (
  stock_table
  .withColumn('date', F.to_date(F.col('date'), 'yyyy-MM-dd'))
  .withColumn('open', F.col('open').cast(DecimalType(18,2)))
  .withColumn('high', F.col('high').cast(DecimalType(18,2)))
  .withColumn('low', F.col('low').cast(DecimalType(18,2)))
  .withColumn('close', F.col('close').cast(DecimalType(18,2)))
)
stock_table.show()

+----------+------+------+------+------+---------+
|      date|  open|  high|   low| close|   volume|
+----------+------+------+------+------+---------+
|2026-02-10|191.38|192.48|188.12|188.54|136764825|
|2026-02-09|184.26|193.66|183.95|190.04|196387351|
|2026-02-06|176.69|187.00|174.60|185.41|231346241|
|2026-02-05|174.93|176.82|171.03|171.88|206312890|
|2026-02-04|179.46|179.58|171.91|174.19|207014116|
|2026-02-03|186.24|186.27|176.23|180.34|202006430|
|2026-02-02|187.20|190.30|184.88|185.61|165794054|
|2026-01-30|191.21|194.49|189.47|191.13|179489463|
|2026-01-29|191.34|193.48|186.06|192.51|171764375|
|2026-01-28|191.27|192.35|189.84|191.52|148552677|
|2026-01-27|187.24|190.00|185.70|188.52|138432307|
|2026-01-26|187.16|189.12|185.99|186.47|124799649|
|2026-01-23|187.50|189.60|186.82|187.67|142748076|
|2026-01-22|184.75|186.17|183.93|184.84|139636626|
|2026-01-21|179.05|185.38|178.40|183.32|200380959|
|2026-01-20|181.90|182.38|177.61|178.07|218355781|
|2026-01-16|189.08|190.44|186.0

In [0]:
stock_table = (
    stock_table
    .sort(F.col('date').desc())
    .dropDuplicates(['date'])
)
display(stock_table)

date,open,high,low,close,volume
2026-02-10,191.38,192.48,188.12,188.54,136764825
2026-02-09,184.26,193.66,183.95,190.04,196387351
2026-02-06,176.69,187.00,174.60,185.41,231346241
2026-02-05,174.93,176.82,171.03,171.88,206312890
2026-02-04,179.46,179.58,171.91,174.19,207014116
2026-02-03,186.24,186.27,176.23,180.34,202006430
2026-02-02,187.20,190.30,184.88,185.61,165794054
2026-01-30,191.21,194.49,189.47,191.13,179489463
2026-01-29,191.34,193.48,186.06,192.51,171764375
2026-01-28,191.27,192.35,189.84,191.52,148552677


In [0]:
stock_table = (
    stock_table
    .dropna(subset=['date', 'volume'])
)

In [0]:
stock_table = (
    stock_table
    .withColumn('month', F.date_format('date', 'MM'))
    .withColumn('year', F.date_format('date', 'yyyy'))
    .withColumn('change', F.col('close') - F.col('open'))
    .withColumn('p_change', F.round(F.col('change')*100 / F.col('open'), 2))
    .withColumn('change_yesterday', F.col('open') - F.lead('close').over(Window.orderBy('date')))
    .withColumn('p_change_yesterday', F.round(F.col('change_yesterday')*100 /  F.lead('close').over(Window.orderBy('date')), 2))
)
display(stock_table)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,open,high,low,close,volume,month,year,change,p_change,change_yesterday,p_change_yesterday
2025-09-18,173.98,177.10,172.96,176.24,191763313,09,2025,2.26,1.30,-2.69,-1.52
2025-09-19,175.77,178.08,175.18,176.67,237182143,09,2025,0.90,0.51,-7.84,-4.27
2025-09-22,175.30,184.55,174.71,183.61,269637001,09,2025,8.31,4.74,-3.13,-1.75
2025-09-23,181.97,182.42,176.21,178.43,192559552,09,2025,-3.54,-1.95,5.00,2.83
2025-09-24,179.77,179.78,175.40,176.97,143564116,09,2025,-2.80,-1.56,2.08,1.17
2025-09-25,174.48,180.26,173.13,177.69,191586733,09,2025,3.21,1.84,-3.71,-2.08
2025-09-26,178.17,179.77,174.93,178.19,148573732,09,2025,0.02,0.01,-3.68,-2.02
2025-09-29,180.43,184.00,180.32,181.85,193063455,09,2025,1.42,0.79,-6.15,-3.30
2025-09-30,182.08,187.35,181.48,186.58,236981032,09,2025,4.50,2.47,-5.16,-2.76
2025-10-01,185.24,188.14,183.90,187.24,173844901,10,2025,2.00,1.08,-3.65,-1.93


In [0]:
stock_table.write.mode('overwrite').saveAsTable('jrvs_dlt.02_silver.stock_data')

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Meta Data Transformations
We want to keep exchange, sector, trading date, adjusted prices...

In [0]:
meta_table = (
    meta_table
    .drop("output_size")
    .crossJoin(company_table.selectExpr(
        "Sector as sector",
        "Currency as currency",
        "Exchange as exchange",
        "AssetType as asset_type"
    ))
)

display(meta_table)

information,symbol,last_refreshed,time_zone,sector,currency,exchange,asset_type
"Daily Prices (open, high, low, close) and Volumes",NVDA,2026-02-10,US/Eastern,TECHNOLOGY,USD,NASDAQ,Common Stock


In [0]:
meta_table = (
    meta_table
    .withColumn('last_refreshed', F.to_date(F.col('last_refreshed'), 'yyyy-MM-dd'))
)
display(meta_table)

information,symbol,last_refreshed,time_zone,sector,currency,exchange,asset_type
"Daily Prices (open, high, low, close) and Volumes",NVDA,2026-02-10,US/Eastern,TECHNOLOGY,USD,NASDAQ,Common Stock


In [0]:
meta_table.write.option('overwriteSchema', True).mode('overwrite').saveAsTable('jrvs_dlt.02_silver.stock_meta_data')

### Company Table (Meta Data)
Metadata specific to the company from which we obtain the stock data.

In [0]:
company_table = (
    company_table.selectExpr(
        "Symbol as symbol",
        "Name as name",
        "CIK as cik",
        "Country as country",
        "Sector as sector",
        "Industry as industry",
        "MarketCapitalization as market_cap",
        "SharesOutstanding as shares_outstanding",
        "Beta as beta",
        "PERatio as pe_ration",
        "ForwardPE as forward_pe",
        "PriceToBookRatio as price_to_book",
        "EVToEBITDA as ev_to_ebitda",
        "AnalystTargetPrice as analyst_target_price",
        "AnalystRatingStrongBuy as analyst_strong_buy",
        "AnalystRatingBuy as analyst_buy",
        "AnalystRatingHold as analyst_hold",
        "AnalystRatingSell as analyst_sell",
        "AnalystRatingStrongSell as analyst_strong_sell",
        "52WeekHigh as 52_week_high",
        "52WeekLow as 52_week_low"
    )
)

display(company_table)


symbol,name,cik,country,sector,industry,market_cap,shares_outstanding,beta,pe_ration,forward_pe,price_to_book,ev_to_ebitda,analyst_target_price,analyst_strong_buy,analyst_buy,analyst_hold,analyst_sell,analyst_strong_sell,52_week_high,52_week_low
NVDA,NVIDIA Corporation,1045810,USA,TECHNOLOGY,SEMICONDUCTORS,4590383137000,24305000000,2.314,46.9,24.51,38.54,38.06,253.62,12,48,3,1,0,212.18,86.6


In [0]:
company_table = (
    company_table
    .withColumn('market_cap', F.col('market_cap').cast(LongType()))
    .withColumn('shares_outstanding', F.col('shares_outstanding').cast(LongType()))
    .withColumn('beta', F.col('beta').cast(DoubleType()))
    .withColumn('pe_ration', F.col('pe_ration').cast(DecimalType(18, 2)))
    .withColumn('forward_pe', F.col('forward_pe').cast(DecimalType(18, 2)))
    .withColumn('price_to_book', F.col('price_to_book').cast(DecimalType(18, 2)))
    .withColumn('ev_to_ebitda', F.col('ev_to_ebitda').cast(DecimalType(18, 2)))
    .withColumn('analyst_target_price', F.col('analyst_target_price').cast(DecimalType(18, 2)))
    .withColumn('analyst_strong_buy', F.col('analyst_strong_buy').cast(IntegerType()))
    .withColumn('analyst_buy', F.col('analyst_buy').cast(IntegerType()))
    .withColumn('analyst_hold', F.col('analyst_hold').cast(IntegerType()))
    .withColumn('analyst_sell', F.col('analyst_sell').cast(IntegerType()))
    .withColumn('analyst_strong_sell', F.col('analyst_strong_sell').cast(IntegerType()))
    .withColumn('52_week_high', F.col('52_week_high').cast(DecimalType(18, 2)))
    .withColumn('52_week_low', F.col('52_week_low').cast(DecimalType(18, 2)))
)

display(company_table)

symbol,name,cik,country,sector,industry,market_cap,shares_outstanding,beta,pe_ration,forward_pe,price_to_book,ev_to_ebitda,analyst_target_price,analyst_strong_buy,analyst_buy,analyst_hold,analyst_sell,analyst_strong_sell,52_week_high,52_week_low
NVDA,NVIDIA Corporation,1045810,USA,TECHNOLOGY,SEMICONDUCTORS,4590383137000,24305000000,2.314,46.90,24.51,38.54,38.06,253.62,12,48,3,1,0,212.18,86.60


In [0]:
company_table.write.option('overwriteSchema', True).mode('overwrite').saveAsTable('jrvs_dlt.02_silver.company_meta_data')